# SHAP Explainer + Global Plots

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TreeExplainer works natively and fast with CatBoost
explainer = shap.TreeExplainer(best_model)

# Use a sample for global plots (SHAP on all 49k test rows is slow and unnecessary)
shap_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42)
shap_values_sample = explainer.shap_values(shap_sample)

print(f"✅ SHAP explainer ready | shap_values shape: {shap_values_sample.shape}")

# Global feature importance (which features matter most, across all farmers)
shap.summary_plot(shap_values_sample, shap_sample, show=False)
plt.tight_layout(); plt.savefig("shap_summary_plot.png", dpi=150); plt.show()

shap.summary_plot(shap_values_sample, shap_sample, plot_type="bar", show=False)
plt.tight_layout(); plt.savefig("shap_bar_plot.png", dpi=150); plt.show()

# Single-Farmer Explanation (Waterfall Plot)

In [ ]:
sample_idx = 0
X_farmer = X_test.iloc[[sample_idx]]

predicted_yield = best_model.predict(X_farmer)[0]
shap_values_farmer = explainer.shap_values(X_farmer)
base_value = explainer.expected_value

print(f"Predicted Yield : {predicted_yield:.2f}")
print(f"Base value      : {base_value:.2f}  (average yield across all farmers)")

explanation = shap.Explanation(
    values=shap_values_farmer[0],
    base_values=base_value,
    data=X_farmer.iloc[0].values,
    feature_names=FEATURES
)

shap.plots.waterfall(explanation, show=False)
plt.tight_layout(); plt.savefig("shap_waterfall_farmer.png", dpi=150); plt.show()

# Recommendation Knowledge Base (direction-aware, not a flat list)

In [ ]:
# Whether "improving" this feature means raising it or lowering it
FEATURE_DIRECTION = {
    "Organic Matter (%)": "higher_is_better",
    "Nitrogen Content (kg/ha)": "higher_is_better",
    "Potassium Content (kg/ha)": "higher_is_better",
    "Soil_Fertility_Index": "higher_is_better",
    "Fertilizer_Consumption": "higher_is_better",
    "Pesticide_Consumption": "lower_is_better",
    "Annual_Rainfall": "higher_is_better",
    "Average_Rainfall": "higher_is_better",
    "Rainy_Months_Count": "higher_is_better",
    "Average_Temperature": "lower_is_better",
    "Temperature_Range": "lower_is_better",
    "Weather_Index": "higher_is_better",
    "Climate_Index": "higher_is_better",
    "Input_Intensity": "higher_is_better",
    "Agricultural_Intensity": "higher_is_better",
}

RECOMMENDATION_DB = {
    "Crop_Encoded": "Consider cultivating crops better suited to the soil and climatic conditions. High-value or climate-resilient crops may improve yield.",
    "Season_Encoded": "Shift the sowing season based on local rainfall and temperature forecasts for better productivity.",
    "Area": "Optimize land utilization through precision farming, proper spacing, and crop planning.",
    "Average_Rainfall": "Use irrigation systems during rainfall deficit periods.",
    "Annual_Rainfall":  "Adopt rainwater harvesting or supplemental irrigation.",
    "Climate_Index":    "Select climate-resilient crop varieties suitable for your region.",
    "Soil_Fertility_Index": "Increase organic manure and balanced fertilizers.",
    "Organic Matter (%)":   "Apply compost or vermicompost.",
    "Nitrogen Content (kg/ha)": "Apply nitrogen fertilizers in split doses.",
    "Potassium Content (kg/ha)":    "Increase potash application.",
    "Average_Temperature":  "Use mulching or shade nets to reduce heat stress.",
    "Temperature_Range":    "Protect crops from temperature fluctuations using covers or tunnels.",
    "Pesticide_Consumption":    "Adopt Integrated Pest Management to reduce pesticide overuse.",
    "Fertilizer_Consumption":   "Apply fertilizers according to soil test recommendations.",
    "Weather_Index":    "Schedule irrigation and sowing using weather forecasts.",
    "Input_Intensity":  "Optimize fertilizer, water and labour input for higher efficiency.",
    "Agricultural_Intensity":   "Practice crop rotation or intercropping for better productivity."
}

# SHAP-Driven Personalized Recommendation Engine

In [ ]:
def generate_recommendations(farmer_row, shap_vals, feature_names, reference_df, top_n=3):
    """Picks the top-N features hurting THIS farmer's yield the most (SHAP < 0)
    and looks up personalized advice — not a fixed rule list."""
    shap_series = pd.Series(shap_vals, index=feature_names)
    negative_features = shap_series[shap_series < 0].sort_values().head(top_n)

    recs = []
    for feat, shap_val in negative_features.items():
        if feat not in RECOMMENDATION_DB:
            continue
        recs.append({
            "feature": feat,
            "farmer_value": farmer_row[feat],
            "median_value": reference_df[feat].median(),
            "shap_contribution": round(float(shap_val), 3),
            "advice": RECOMMENDATION_DB[feat],
        })
    return recs

recs = generate_recommendations(
    farmer_row=X_farmer.iloc[0],
    shap_vals=shap_values_farmer[0],
    feature_names=FEATURES,
    reference_df=X_train,
    top_n=3,
)

for r in recs:
    print(f"✗ {r['feature']} (SHAP {r['shap_contribution']}): {r['advice']}")

# Full Dashboard-Style Report

In [ ]:
# ==============================================================================
# 🌾 EXPLAINABLE AI DECISION SUPPORT LAYER (Farmer-Facing)
# ------------------------------------------------------------------------------
# Does NOT retrain the model, does NOT touch the SHAP explainer.
# Reuses: best_model, explainer, X_train, X_test, FEATURES (must already exist
# in your notebook from your CatBoost + SHAP cells above).

import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# 1. HUMAN-READABLE LABELS — one line of farmer language per raw feature name
# ------------------------------------------------------------------------------
FEATURE_LABELS = {
    "Crop_Year":                 "Crop year / timing",
    "Area":                      "Land utilization",
    "Season_Encoded":            "Sowing season",
    "Crop_Encoded":              "Crop selection",
    "State_Name_Encoded":        "Regional growing conditions",
    "District_Name_Encoded":     "Local growing conditions",
    "pH Level":                  "Soil pH balance",
    "Organic Matter (%)":        "Soil organic matter",
    "Nitrogen Content (kg/ha)":  "Nitrogen availability",
    "Phosphorus Content (kg/ha)":"Phosphorus availability",
    "Potassium Content (kg/ha)": "Potassium availability",
    "Soil_Fertility_Index":      "Overall soil fertility",
    "Soil_Type_Encoded":         "Soil type suitability",
    "Fertilizer_Consumption":    "Fertilizer usage",
    "Pesticide_Consumption":     "Pesticide usage",
    "Annual_Rainfall":           "Annual rainfall",
    "Average_Rainfall":          "Seasonal rainfall",
    "Rainy_Months_Count":        "Rainy season duration",
    "Average_Temperature":       "Average temperature",
    "Temperature_Range":         "Temperature stability",
    "Weather_Index":             "Overall weather favourability",
    "Climate_Index":             "Overall climate suitability",
    "Input_Intensity":           "Agricultural input levels",
    "Agricultural_Intensity":    "Farming intensity",
}

def humanize(feature_name):
    """Fallback for any feature not explicitly mapped above."""
    return FEATURE_LABELS.get(
        feature_name,
        feature_name.replace("_", " ").replace("Encoded", "").strip().capitalize()
    )

# ------------------------------------------------------------------------------
# 2. DIRECTION MAP — tells the simulator how to nudge a feature toward a
#    "better" agronomic value using real percentiles from X_train (no hardcoded
#    numbers). Categorical/encoded and non-adjustable features are left out on
#    purpose -- they're still explained to the farmer, just not simulated.
# ------------------------------------------------------------------------------
FEATURE_DIRECTION = {
    "Organic Matter (%)":         "increase",
    "Nitrogen Content (kg/ha)":   "increase",
    "Phosphorus Content (kg/ha)": "increase",
    "Potassium Content (kg/ha)":  "increase",
    "Soil_Fertility_Index":       "increase",
    "Fertilizer_Consumption":     "increase",
    "Annual_Rainfall":            "increase",
    "Average_Rainfall":           "increase",
    "Rainy_Months_Count":         "increase",
    "Weather_Index":              "increase",
    "Climate_Index":              "increase",
    "Input_Intensity":            "increase",
    "Agricultural_Intensity":     "increase",
    "pH Level":                   "context",   # too acidic or too alkaline both hurt
    "Average_Temperature":        "context",   # too hot or too cold both hurt
    "Temperature_Range":          "decrease",  # large swings = more stress
    "Pesticide_Consumption":      "context",   # over- or under-use both hurt
}

# ------------------------------------------------------------------------------
# 3. RECOMMENDATION ENGINE — specific, actionable text per feature.
#    "low"/"high"/"generic" variants are chosen based on where the farmer's
#    actual value sits relative to the X_train distribution for that feature.
# ------------------------------------------------------------------------------
RECOMMENDATIONS = {
    "Nitrogen Content (kg/ha)":   {"low": "Apply nitrogen fertilizer in split doses"},
    "Phosphorus Content (kg/ha)": {"low": "Apply phosphorus-rich fertilizer (e.g. DAP) based on soil test"},
    "Potassium Content (kg/ha)":  {"low": "Apply potash fertilizer to boost potassium levels"},
    "Organic Matter (%)":         {"low": "Add compost or farmyard manure"},
    "Soil_Fertility_Index":       {"low": "Improve fertility using crop rotation and green manure"},
    "pH Level":                   {"low": "Apply agricultural lime to raise soil pH",
                                    "high": "Apply gypsum or organic matter to lower soil pH"},
    "Fertilizer_Consumption":     {"low": "Apply fertilizer according to soil test recommendations"},
    "Pesticide_Consumption":      {"low": "Apply need-based pesticide protection to control losses",
                                    "high": "Reduce pesticide use and adopt integrated pest management"},
    "Annual_Rainfall":            {"low": "Install drip irrigation to supplement low rainfall"},
    "Average_Rainfall":           {"low": "Install drip irrigation during dry periods"},
    "Rainy_Months_Count":         {"low": "Plan supplemental irrigation for shorter rainy seasons"},
    "Average_Temperature":        {"low": "Use row covers or greenhouses to retain warmth",
                                    "high": "Use mulching and shade nets to reduce heat stress"},
    "Temperature_Range":          {"low": "Maintain consistent field conditions",
                                    "high": "Use mulching to stabilize soil temperature swings"},
    "Weather_Index":              {"low": "Adjust sowing schedule using weather forecasts"},
    "Climate_Index":              {"low": "Choose climate-resilient crop varieties"},
    "Input_Intensity":            {"low": "Increase balanced use of fertilizer, water, and inputs"},
    "Agricultural_Intensity":     {"low": "Adopt improved agronomic practices to raise farming intensity"},
    "Crop_Encoded":               {"generic": "Select a crop suitable for local climate and soil"},
    "Season_Encoded":             {"generic": "Shift sowing season based on rainfall pattern"},
    "Area":                       {"generic": "Optimize land utilization and field layout"},
    "State_Name_Encoded":         {"generic": "Consult local agricultural extension services for regional best practices"},
    "District_Name_Encoded":      {"generic": "Consult local agricultural extension services for district-specific advice"},
    "Soil_Type_Encoded":          {"generic": "Adapt farming practices to your soil type"},
}

POSITIVE_PHRASES = {
    "Fertilizer_Consumption":     "Good fertilizer usage",
    "Pesticide_Consumption":      "Balanced pesticide usage",
    "Annual_Rainfall":            "Suitable rainfall",
    "Average_Rainfall":           "Suitable rainfall",
    "Rainy_Months_Count":         "Well-distributed rainy season",
    "pH Level":                   "Healthy soil pH",
    "Organic Matter (%)":         "Healthy soil organic matter",
    "Nitrogen Content (kg/ha)":   "Balanced nitrogen availability",
    "Phosphorus Content (kg/ha)": "Balanced phosphorus availability",
    "Potassium Content (kg/ha)":  "Balanced nutrient availability",
    "Soil_Fertility_Index":       "Healthy soil fertility",
    "Average_Temperature":        "Suitable temperature",
    "Temperature_Range":          "Stable temperature conditions",
    "Weather_Index":              "Favourable weather",
    "Climate_Index":              "Favourable climate conditions",
    "Input_Intensity":            "Good input management",
    "Agricultural_Intensity":     "Strong farming intensity",
    "Crop_Encoded":               "Good crop selection",
    "Season_Encoded":             "Optimal sowing season",
    "Area":                       "Efficient land utilization",
}

# ------------------------------------------------------------------------------
# 4. CORE HELPERS
# ------------------------------------------------------------------------------
def _level(feature, value, X_train):
    """Return 'low' or 'high' based on where `value` sits vs the X_train median
    for that feature. Used to pick the right recommendation/positive phrase."""
    median = X_train[feature].median()
    return "low" if value <= median else "high"

def get_recommendation(feature, value, X_train):
    entry = RECOMMENDATIONS.get(feature)
    if entry is None:
        return None
    if "generic" in entry:
        return entry["generic"]
    level = _level(feature, value, X_train)
    return entry.get(level, entry.get("low"))

def get_positive_phrase(feature):
    return POSITIVE_PHRASES.get(feature, f"Favourable {humanize(feature).lower()}")

def simulate_improved_value(feature, current_value, X_train):
    """Model-driven improvement: nudge the feature toward a realistic better
    value using actual X_train percentiles for that feature -- never a
    hardcoded number, and never applied to non-simulatable (categorical)
    features."""
    direction = FEATURE_DIRECTION.get(feature)
    series = X_train[feature]
    if direction == "increase":
        target = series.quantile(0.75)
        return max(current_value, target)
    elif direction == "decrease":
        target = series.quantile(0.25)
        return min(current_value, target)
    elif direction == "context":
        median = series.median()
        return current_value + (median - current_value) * 0.5  # halfway toward median
    return current_value  # not simulatable -> leave unchanged

# ------------------------------------------------------------------------------
# 5. MAIN DECISION-SUPPORT FUNCTION
# ------------------------------------------------------------------------------
def generate_decision_support(row_index, top_n=5):
    """
    Builds a full farmer-facing Decision Support report for one row of X_test.
    Reuses the already-trained `best_model` and `explainer` -- no retraining.
    """
    instance = X_test.iloc[[row_index]][FEATURES]

    # --- SHAP values for this single instance (reuses existing explainer) ---
    shap_row = explainer(instance)
    shap_vals = shap_row.values[0]
    base_value = shap_row.base_values[0] if np.ndim(shap_row.base_values) else shap_row.base_values

    current_pred = float(best_model.predict(instance)[0])

    contrib_df = pd.DataFrame({
        "Feature": FEATURES,
        "Value": instance.iloc[0].values,
        "SHAP": shap_vals,
    }).sort_values("SHAP")

    negative_factors = contrib_df[contrib_df["SHAP"] < 0].sort_values("SHAP").head(top_n)
    positive_factors = contrib_df[contrib_df["SHAP"] > 0].sort_values("SHAP", ascending=False).head(top_n)

    # --- Build recommendations for negative factors only ---
    recommendations = []
    for _, r in negative_factors.iterrows():
        rec = get_recommendation(r["Feature"], r["Value"], X_train)
        if rec and rec not in recommendations:
            recommendations.append(rec)

    # --- Simulate a realistic improvement: nudge ONLY negatively-contributing,
    #     simulatable features toward better agronomic values, then re-run the
    #     already-trained model (no retraining) ---
    improved_instance = instance.copy()
    for _, r in negative_factors.iterrows():
        feat = r["Feature"]
        if feat in FEATURE_DIRECTION:
            improved_instance.at[improved_instance.index[0], feat] = simulate_improved_value(
                feat, r["Value"], X_train
            )
    improved_pred = float(best_model.predict(improved_instance)[0])
    pct_change = ((improved_pred - current_pred) / current_pred * 100) if current_pred != 0 else 0.0

    return {
        "current_pred": current_pred,
        "improved_pred": improved_pred,
        "pct_change": pct_change,
        "positive_factors": positive_factors,
        "negative_factors": negative_factors,
        "recommendations": recommendations,
    }

# ------------------------------------------------------------------------------
# 6. DASHBOARD PRINTER
# ------------------------------------------------------------------------------
def print_dashboard(result):
    print("=" * 50)
    print("🌾 AI Crop Yield Decision Support")
    print("=" * 50)

    print("\nPredicted Yield\n")
    print(f"{result['current_pred']:.2f} t/ha")

    print("\nTop Positive Factors\n")
    if len(result["positive_factors"]) == 0:
        print("  (none identified for this field)")
    for _, r in result["positive_factors"].iterrows():
        print(f"✔ {get_positive_phrase(r['Feature'])}")

    print("\nTop Factors Reducing Yield\n")
    if len(result["negative_factors"]) == 0:
        print("  (none identified for this field)")
    for _, r in result["negative_factors"].iterrows():
        print(f"❌ {humanize(r['Feature'])} is reducing productivity")

    print("\nRecommendations\n")
    if len(result["recommendations"]) == 0:
        print("  (no specific action needed)")
    for rec in result["recommendations"]:
        print(f"✔ {rec}")

    print("\nPotential Yield After Improvement\n")
    print(f"{result['current_pred']:.2f} t/ha")
    print("        ↓")
    print(f"{result['improved_pred']:.2f} t/ha")
    print(f"({result['pct_change']:+.0f}%)")

    print("\n" + "=" * 50)

# ------------------------------------------------------------------------------
# 7. RUN IT — pick any row from X_test (e.g. the first one) as a demo farmer
# ------------------------------------------------------------------------------
result = generate_decision_support(row_index=0, top_n=5)
print_dashboard(result)

# Cat Boost Regressor 

In [ ]:
# ==============================================================================
# TRAIN & SAVE THE BEST MODEL (CatBoost — consistently highest R2 across every
# stage in your results table) ON Final_Crop_Yield_Dataset.csv
# ==============================================================================
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -------------------------------------------------
# 1. LOAD (no need to re-run the merge cell)
# -------------------------------------------------
final_df = pd.read_csv("ML_Dataset_Featured/Final_Crop_Yield_Dataset.csv")
print(f"✅ Loaded Final_Crop_Yield_Dataset.csv | Shape: {final_df.shape}")

# -------------------------------------------------
# 2. ENCODE THE ONE REMAINING TEXT COLUMN
# -------------------------------------------------
# Season/Crop/State_Name/District_Name are already encoded in this file, but
# "Soil Type" is still raw text ('Black lava soil', 'Alkaline', 'Loamy', 'Clay').
le_soil = LabelEncoder()
final_df["Soil_Type_Encoded"] = le_soil.fit_transform(final_df["Soil Type"].astype(str))

# -------------------------------------------------
# 3. FULL ("STAGE 7 / FINAL DATASET") FEATURE SET
# -------------------------------------------------
TARGET = "Yield"
FEATURES = [
    "Crop_Year", "Area", "Season_Encoded", "Crop_Encoded",
    "State_Name_Encoded", "District_Name_Encoded",
    "pH Level", "Organic Matter (%)", "Nitrogen Content (kg/ha)",
    "Potassium Content (kg/ha)", "Soil_Fertility_Index", "Soil_Type_Encoded",
    "Fertilizer_Consumption", "Pesticide_Consumption",
    "Annual_Rainfall", "Average_Rainfall", "Rainy_Months_Count",
    "Average_Temperature", "Temperature_Range",
    "Weather_Index", "Climate_Index", "Input_Intensity", "Agricultural_Intensity",
]

model_ready_df = final_df.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
print(f"✅ Model-ready rows: {len(model_ready_df):,} / {len(final_df):,}")

X = model_ready_df[FEATURES]
y = np.log1p(model_ready_df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ Train: {len(X_train):,} | Test: {len(X_test):,}")

# -------------------------------------------------
# 4. TRAIN CATBOOST (best of the three per your results)
# -------------------------------------------------
best_model = CatBoostRegressor(
    iterations=300, learning_rate=0.05, depth=6, random_state=42, verbose=False
)
best_model.fit(X_train, y_train)

pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
pred = np.expm1(pred)
print(f"✅ MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")
print("   (Expect this to land close to your Stage 7 CatBoost R2 of 0.7653 —")
print("    not higher, since this is the same 'final dataset' training Stage 7 already did.)")

# -------------------------------------------------
# 5. SAVE THE MODEL (+ everything needed to reuse it later)
# -------------------------------------------------
best_model.save_model("best_catboost_model.cbm")          # CatBoost's native format (recommended)
joblib.dump(best_model, "best_catboost_model.joblib")      # generic sklearn-style pickle, optional
joblib.dump(le_soil, "soil_type_label_encoder.joblib")     # needed to encode Soil Type at inference time
joblib.dump(FEATURES, "model_feature_list.joblib")         # needed to build inputs in the right column order

print("💾 Saved: best_catboost_model.cbm")
print("💾 Saved: best_catboost_model.joblib")
print("💾 Saved: soil_type_label_encoder.joblib")
print("💾 Saved: model_feature_list.joblib")